<a href="https://colab.research.google.com/github/Jgriffiths92/Open_E-Dope/blob/master/Kivy_App_To_APK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Open_E-Dope APK Compiler Notebook
This notebook automates system dependency installation, repository fetching, Buildozer configuration, and APK compilation for the **Open_E-Dope** application.

### Step 1: Install Ubuntu System Dependencies
Buildozer requires standard Linux development headers, utilities, and **Java 17** to cross-compile the Android toolchain.

In [ ]:
# Update apt registries
!sudo apt-get update

# Install essential packages and Java 17
# Install essential packages, Java 17, and Python 3.11 system bin targets
!sudo apt-get install -y \
    build-essential \
    uuid-dev \
    zlib1g-dev \
    libncurses5-dev \
    libgdbm-dev \
    libnss3-dev \
    libssl-dev \
    libsqlite3-dev \
    libreadline-dev \
    libffi-dev \
    libbz2-dev \
    liblzma-dev \
    tk-dev \
    libgdbm-compat-dev \
    gawk \
    unzip \
    zip \
    autoconf \
    automake \
    libtool \
    pkg-config \
    cmake \
    git \
    openjdk-17-jdk \
    python3.11 \
    python3.11-dev \
    python3.11-distutils

!java -version

### Step 2: Install Buildozer and Pin Cython
Pinning `Cython==0.29.36` avoids compatibility issues in the Android build toolchain.

In [ ]:
%pip install --upgrade buildozer Cython==0.29.36

### Step 3: Clone the Open_E-Dope Repository
This pulls down the project, including its pre-configured `buildozer.spec` file.

In [ ]:
import os

if not os.path.exists('Open_E-Dope'):
    !git clone https://github.com/Jgriffiths92/Open_E-Dope.git

%cd Open_E-Dope

### Step 4: Pin the Python Recipe Version
Set the `python3` requirement to Python 3.11 while preserving the other application requirements.

In [ ]:
from pathlib import Path
import re

spec_path = Path('buildozer.spec')
content = spec_path.read_text()
match = re.search(r'(?m)^(\s*requirements\s*=\s*)(.*)$', content)
if match is None:
    raise ValueError('Could not find the requirements setting in buildozer.spec')

requirements = [item.strip() for item in match.group(2).split(',') if item.strip()]
requirements = [
    item for item in requirements
    if item.split('==', 1)[0].strip().lower() not in {'python3', 'hostpython3'}
]
requirements.insert(0, 'python3==3.11')
content = content[:match.start()] + match.group(1) + ', '.join(requirements) + content[match.end():]
spec_path.write_text(content)
print('Successfully set the python3 requirement to Python 3.11.')

### Step 5: Build the APK
Buildozer downloads the Android SDK/NDK, processes the custom Java source files, and packages the application. The repository's `buildozer.spec` already enables automatic acceptance of the Android SDK license.

In [ ]:
# Clean previous build artifacts, then compile
!buildozer android clean && buildozer -v android debug

### Step 6: Locate your APK
After compilation, check the `bin/` directory for the generated APK.

In [ ]:
!ls -lah bin/